# 02 — Data Preparation for Qwen3-VL Fine-Tuning

Converts Gemini auto-labels into conversation-format training data
for Qwen3-VL QLoRA fine-tuning.


In [ ]:
# Cell 1: Load labels and build conversation pairs
import json, glob, random
from pathlib import Path

LABELS_DIR = Path('/content/drive/MyDrive/numera-ml/data/labels')
IMAGES_DIR = Path('/content/drive/MyDrive/numera-ml/data/page_images')
OUTPUT_DIR = Path('/content/drive/MyDrive/numera-ml/data/training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

label_files = sorted(glob.glob(str(LABELS_DIR / '**/*.json'), recursive=True))
print(f'Found {len(label_files)} labeled pages')

# Build conversation-format training data
conversations = []
for lf in label_files:
    label = json.loads(Path(lf).read_text())
    img_path = str(IMAGES_DIR / Path(lf).relative_to(LABELS_DIR).with_suffix('.png'))

    if not Path(img_path).exists():
        continue

    conversations.append({
        'image': img_path,
        'conversations': [
            {'role': 'user', 'content': 'Extract all financial tables from this page with complete structure, zone classification, periods, currency, and values. Return JSON.'},
            {'role': 'assistant', 'content': json.dumps(label)},
        ]
    })

print(f'Built {len(conversations)} training conversations')


In [ ]:
# Cell 2: Train/val/test split (80/10/10)
random.shuffle(conversations)

n = len(conversations)
train = conversations[:int(n * 0.8)]
val = conversations[int(n * 0.8):int(n * 0.9)]
test = conversations[int(n * 0.9):]

for name, data in [('train', train), ('val', val), ('test', test)]:
    path = OUTPUT_DIR / f'{name}.json'
    path.write_text(json.dumps(data, indent=2))
    print(f'{name}: {len(data)} samples → {path.name}')
